# <strong>Man-in-the-Middle (MITM) Attacks</strong>

 The purpose of this lab is to introduce you to the Domain Name System (DNS), its vulnerabilities, and Domain Name System Security (DNSSEC). It will take you through a hands-on demonstration of one form of attack, and have you implement DNSSEC in order to mitigate it. You need a working knowledge of TCP/IP, the client-server model, and the Unix command line. Experience with dns, tcpdump, ettercap, and public key cryptography is also helpful.

After successfully completing this lab, you should be able to:

1. Understand the basics of DNS
2. Understand the basic DNS vulnerabilities
3. Understand the basics and vulnerable nature of ARP
4. Understand the methodology of man-in-the-middle attacks
5. Understand the basics of public key cryptography
6. Understand the goal and limitations of cryptographic authentication
7. Understand the necessity of DNSSEC
8. Understand the basics of DNSSEC

Before you begin this lab, you may view the readings below to understand the basics of DNS, its vulnerabilities, and more.

In [1]:
# Click on the topics below to read more about them.
import ipywidgets as widgets
from IPython.display import display
from IPython.display import HTML as DHTML

# DNS Overview
dns_overview_content = """
<b>DNS Overview:</b><br>
For this exercise, it's important to understand the basic DNS lookup process. An end-user starts by querying a DNS cache on their local network. For example, the end-user's web browser might generate a request for a specified address (e.g. "www.google.com.") and forwards it to the local DNS cache. The DNS cache is a semi-public server, typically maintained by the end-user's ISP or organization. If the local cache doesn't know the answer, it recursively traverses the DNS tree until it resolves the original request of the end-user. The local cache must be configured with the addresses for the DNS root servers. The cache can then ask a root server for "www.google.com.". The root does not know the answer, but can refer the local cache to the "com." servers. The cache can then ask the a ".com" server for "www.google.com.". The ".com" server does not know the answer either, but it can refer the cache to the servers for "google.com." Finally, the local cache queries a server for "google.com." and this server returns the desired answer. The server is called a cache because, for efficiency reasons, it will store answers for a pre-determined amount of time so that it can answer without consulting the tree if a query for the same information comes in.
"""

# DNS Vulnerabilities
dns_vulnerabilities_content = """
<b>DNS Vulnerabilities:</b><br>
Like most protocols designed for the early internet, DNS wasn't designed with any security concerns in mind. Classical DNS is done entirely via plaintext messages, meaning that a third party can easily read the conversation. It also does very little by way of message verification, meaning that the end-user has no way of knowing if the message was altered en-route or if DNS servers supplying the information are acting maliciously. This leaves the service open to several well-known vectors of attack which will be summarized in the following sections.<br><br>

It should be noted that most classic DNS caches have been patched to include some safeguards. Most commonly, they randomize the sending port when making their queries to outside servers and also randomize a field in the DNS protocol called the "Query ID" which was originally meant to uniquely identify queries using the same port. They then verify that those fields in the received response match the originals. This makes it harder for a remote attacker to forge a response as the attacker must guess the port and the Query ID. However, an attacker that can observe the original query learns both the port and Query ID and can trivially forge a reply.
"""

dns_overview = widgets.HTML(value=dns_overview_content)
dns_vulnerabilities = widgets.HTML(value=dns_vulnerabilities_content)

dns_subaccordion = widgets.Accordion(children=[dns_overview, dns_vulnerabilities])
dns_subaccordion.set_title(0, "DNS Overview")
dns_subaccordion.set_title(1, "DNS Vulnerabilities")

# ARP Overview
arp_overview_content = """
<b>ARP Overview:</b><br>
<p>
  ARP is another protocol developed for basic networking. ARP is important to this exercise because you'll be using a flaw in the protocol to position yourself for the man-in-the-middle attack. ARP is used to resolve a network layer address (e.g., IP address) to a link layer address (e.g., MAC address). When a host wishes to send an IP packet to a remote host, it first creates the IP packet. However, the hardware in the network interface must encapsulate the packet in a link layer frame (commonly Ethernet) before sending the data out on the wire. In order to assign the frame to the correct destination address, the local host must use ARP to determine which hardware address corresponds to the next hop for the IP packet. This involves sending special ARP messages called ARP Requests on the local Ethernet and receiving replies from hosts that have the answer.
</p>

<p>
  For example, the host 10.10.10.20 needs to send data to the Internet. In order to reach the Internet, it must send the data through its default gateway, which is configured with IP address 10.10.10.10. The sending host 10.10.10.20 needs to know what Ethernet address 10.10.10.10 has in order to put the correct MAC address in the frame header. In order to discover this, 10.10.10.20 needs to broadcast an ARP request to all Ethernet nodes:
</p>

<pre><code>arp who-has 10.10.10.10 tell 10.10.10.20</code></pre>

<p>
  ... Hopefully, 10.10.10.20 will receive a reply like:
</p>

<pre><code>arp reply 10.10.10.10 is-at 2e:2f:30:31:32:33</code></pre>

<p>
  ... from some host that knows the answer. Host 10.10.10.20 will update its ARP tables associating the IP 10.10.10.10 with the MAC address 2e:2f:30:31:32:33 and won't ask again for some time.
</p>
"""

# ARP Spoofing
arp_spoofing_content = """
<b>ARP Spoofing:</b><br>
<p>
ARP Spoofing is a technique for redirecting traffic that can allow an arbitrary interface on an otherwise unsecured switched network to receive traffic that would not normally be sent to it.
</p>
<p>
ARP Spoofing relies on the decentralized, unauthenticated, and completely trusting nature of ARP to succeed. In ARP Spoofing, the attacker (or spoofer) tells specific hosts or entire networks that its MAC address should be associated with the victim's IP address, rather than the correct MAC to IP mapping. This is commonly used to tell network hosts that an attacker is the default gateway instead of the real default gateway. The spoofer keeps the victim's real MAC address in its ARP table, and prepares to receive data from network hosts.
</p>
<p>
When an attacker pretends to be the default gateway for a network, all hosts that received the bogus ARP messages now believe that the spoofer's MAC should be inserted into any Ethernet frame destined for the Internet or another IP subnet instead of the real default gateway's MAC address. This is especially meaningful because Ethernet switches build their own ARP tables in order to efficiently direct traffic and avoid broadcasting like a hub. This means that Ethernet switches will be fooled in addition to the remote hosts. As a result, switches will direct default gateway-bound traffic to the spoofer, not the real default gateway. Furthermore, a spoofing host that wishes to remain undetected can rewrite and forward the Ethernet frames to the real default gateway. This method silently maintains the victim's connections while routing them conveniently under the nose of the sniffer. In this way, ARP Spoofing acts as a kind of network reconfiguration at the data link layer and highlights Ethernet's dependence on ARP.
</p>
<p>
At this point, the spoofing host has almost total control of the network traffic and can see and sniff arbitrary hosts' data with the use of a promiscuous interface and packet sniffing software. ARP Spoofing does slow down network traffic because of the additional hops and processing involved, but the clever attacker will target only the hosts she wants to affect, rather than the whole network. ARP Spoofing also requires multiple spoofing actions to maintain the tables in their bogus state, but this is not a difficult burden to bear. Furthermore, the careful spoofer can restore the ARP tables of the network before quitting, resulting in minimal network disruption due to the spoofing.
</p>
<p>
While there are strategies against ARP Spoofing, notably ARP monitoring, fixed ARP tables and VLANs, these are expensive in terms of maintenance, network design, and may require specialized hardware. Because of this, most networks in use today are at least partially vulnerable to ARP Spoofing attacks.
</p>
<p>
Being the "man in the middle" gives you the opportunity to perform the DNS manipulation discussed earlier.
</p>
"""

# Create the widgets
arp_overview = widgets.HTML(value=arp_overview_content)
arp_spoofing = widgets.HTML(value=arp_spoofing_content)

# Add them to an Accordion
arp_subaccordion = widgets.Accordion(children=[arp_overview, arp_spoofing])
arp_subaccordion.set_title(0, "ARP Overview")
arp_subaccordion.set_title(1, "ARP Spoofing")

# To display in notebook
arp_subaccordion


# Main topics
mitm = widgets.HTML(value="""
<b>Man in the Middle:</b><br>
The man-in-the-middle attack vector is fairly common to network services, and effective on virtually all communications that don't make use of cryptographic encryption or authentication. In essence, the attacker must position themself between the client and the server on a network. From there, they can intercept messages going in either direction and alter or deny them at will, violating either the integrity or the availability of the service. In the context of DNS, it is most common for the attacker to position themself somewhere between the end-user and the relevant DNS server, then alter the DNS response to supply a different IP address for the requested internet address, effectively rerouting the user anywhere they want. There is very little that can be done to mitigate this attack using regular DNS protocol besides protecting the network path taken by the DNS messages, and it will also be the form of attack highlighted by this exercise.
""")

birthday_attack = widgets.HTML(value="""
<b>Birthday Attack:</b><br>
Unlike man-in-the-middle, the Birthday attack is specific to DNS and has the goal of "cache poisoning." Most DNS information remains in the DNS cache for somewhere between an hour and a day after it is initially requested. While this is important for efficiency (a cache lookup is much less expensive in terms of time and resources than a recursive query to the outside Internet), it means that if an attacker can supply the cache with malicious but seemingly valid data for a common address, they can redirect a potentially large number of users in the subsequent time-frame. It's worth noting that there are several other forms of cache poisoning that involve inserting unrequested data into other fields of the DNS response, but they use a more systematic approach that is completely ineffective against any up-to-date caching server.<br><br>

The Birthday attack uses a comparatively simple means to this end. An attacker will send a query to the caching server quickly followed by a response to that same query. If the response appears valid to the caching sever, it will accept it and add the "poison" data into cache. Virtually all caching servers have a policy of ignoring any subsequent responses received for the same query. However, to make the faked response seem valid, the attacker may have to do several things including: spoofing the ip of the DNS server queried by the cache, correctly guessing the sending port used by the cache, and correctly guessing the query id used by the cache. The later two require correctly guessing 16-bit numbers(both of which are ideally random) which gives the attack a low, but non-negligible, chance of success.
""")

kaminsky_attack = widgets.HTML(value="""
<b>Kaminsky Attack:</b><br>
The Kaminsky attack is a well-known derivative of the birthday attack that uses various means to increase success probability, the specifics of which require a fairly detailed understanding of certain edge-cases in the DNS protocol.
""")

pk_crypto = widgets.HTML(value="""
<b>Public Key Cryptography:</b><br>
Public key cryptography is built on the concept of a "one-way", also called "asynchronous," function. Mathematically, it's trivial to get from point A to point B, but extremely difficult if not impossible to get from point B back to point A. The asynchronous function is essential because it allows for cryptography without the secure exchange of a shared key ahead of time.<br><br>

In the public key system, a user generates what's referred to as an "asynchronous key pair." This pair includes a private key that is very hard to derive without previously knowing it, and a public key which is freely distributed to other users. The private key, by virtue of the chosen cryptographic method, can be used to sign any message. The owner of the private key would then distribute the message and the generated signature. Any user with the public key, message, and signature can then verify that the signature matches the message and was generated by the private key.
""")

crypto_auth = widgets.HTML(value="""
<b>Cryptographic Authentication:</b><br>
Cryptographic Authentication refers to the process of generating a digital signature(similar to a hash) for a given message. An end-user can use the message, signature, and public key of the sender to verify that the message hasn't been changed and was indeed signed by the sender (as described in the previous section). Provided that the public-key is trusted, either because it was also signed, by the previous DNS server in the chain, or was for a root server, the end-user has cryptographic proof of the sender and data-integrity.
""")

dnssec = widgets.HTML(value="""
<b>DNSSEC:</b><br>
DNSSEC is an enhancement to the DNS protocol that integrates cryptographic authentication into DNS messages. DNSSEC assures data integrity, mitigating man in the middle and cache-poisoning attacks using a "chain of trust." In essence, you trust the response you get from each server if it has been vouched for by the server before it in the lookup. It's still necessary to trust the supplied public key for the root DNS servers, but there are various political and administrative procedures in place to ensure their trustability.
""")

tcpdump = widgets.HTML(value="""
<b>TCPDUMP:</b><br>
To examine the MAC addresses on packets in this exercise and possibly the contents of the packets themselves, we recommend you use a command-line tool called TCPDUMP. It's a very common and useful tool for looking at network traffic so you'll want to get acquainted with the man-page if you aren't already.
""")

# Main accordion
main_accordion = widgets.Accordion(children=[
    dns_subaccordion,
    mitm,
    birthday_attack,
    kaminsky_attack,
    pk_crypto,
    crypto_auth,
    dnssec,
    arp_subaccordion,
    tcpdump
])

main_titles = [
    "Domain Name System",
    "Man in the Middle",
    "Birthday Attack",
    "Kaminsky Attack",
    "Public Key Cryptography",
    "Cryptographic Authentication",
    "Domain Name System Security Extensions (DNSSEC)",
    "Address Resolution Protocol (ARP)",
    "TCPDUMP"
]

for i, title in enumerate(main_titles):
    main_accordion.set_title(i, title)


display(main_accordion)


Accordion(children=(Accordion(children=(HTML(value='\n<b>DNS Overview:</b><br>\nFor this exercise, it\'s impor…

### Step 0: Starting the Lab

Click the button to begin creating the experiment.

<strong>Note:</strong> If your buttons are not displaying, click on the <img width='20px' height='20px' style='margin-left: 1px;' src='resources/fast_forward.png'> icon at the top of your notebook to render all widgets.

In [2]:
# Click the button below to start your lab.
import os
import subprocess
import re
import threading
import queue
import time
import sys
from IPython.display import display, HTML
import ipywidgets as widgets
import logging

# This is a troublesome file that will throw unnecessary warnings and other errors.
# Delete it if it exists. (Wasn't an issue in older Jupyter versions.)
# !(rm -f ~/.local/share/jupyter/nbsignatures.db)

# Global variable for the checker script.
runAllSteps = False

# Adding the "resource/" directory so that we can import the start.py file.
module_dir = os.path.join(os.getcwd(), 'resources')
if module_dir not in sys.path:
    sys.path.append(module_dir)

# Importing the prepare_lab function.
from functions import *

# Defining some stuff for the output below.
output0 = widgets.Output()
startButton = widgets.Button(description="Start Lab")
labname = "mitm"

# Defining the button handler for startButton.
def on_start_clicked(b):
    prepare_lab(labname, output0)

# Providing on_click functionality.
startButton.on_click(on_start_clicked)

# Display button and output area.
display(startButton, output0)

Button(description='Start Lab', style=ButtonStyle())

Output()

<hr>

If you previously stopped your lab, you may restore your progress below by clicking "Load Lab". <u>You do not have to load your lab if you signed out, closed your notebook, or exited your node(s) or XDC by using ```exit```.</u>

In [3]:
# Click the button below to load your lab.
def loadlab(b):
    load_lab(labname, output0_2)

# Creating the button.
loadButton = widgets.Button(description="Load Lab")

# Creating an output area.
output0_2 = widgets.Output()

# Run the command on click.
loadButton.on_click(loadlab)

# Display the output.
display(loadButton, output0_2)

Button(description='Load Lab', style=ButtonStyle())

Output()

## <strong>Introduction</strong>

The network in this exercise is designed to simulate an end-user and a recursive DNS system as simply as possible. To that end, there is a single authoritative DNS server which handles all the levels in the setup (root, com., google.com.). The DNS caching server has a direct line to the authoritative DNS server and shares a switch with the end-user and attacker on it's other interface. Ordinarily, when the end-user makes a query, it will be sent to the caching server which then traverses the small DNS tree in the setup by making several queries to the authoritative server. 

Once it gets the required response, it forwards it on to the end-user. The caching server also stores the answer for a small (10 seconds) but easily observable amount of time. For more detailed information, it's easiest to consult the actual simulation once you've set it up. 

Upon starting your lab, you will have a topology that looks like this.

<div style="text-align: center; padding-right: 40px"><img src="resources/mitm/topo.png"></div>

<strong>This lab will contain four topics:</strong>
1. Understanding DNS
2. Understanding Network Topology
3. Using Ettercap
4. Implementing DNSSEC

## <strong>Topic 1: Understanding DNS</strong>

For this topic, you are going to use the `dig` command, and interpret its response. `dig` is a standard command line tool for making DNS queries (typically for network debugging purposes). In this exercise, you'll be using it both to simulate DNS queries from an internet browser and to debug your configuration if necessary. You should be familiar with its standard Unix syntax.

### Step 1: Using `dig`

<strong>Your task:</strong> Login to the `client` machine and use this `dig` command to lookup the IPv4 address of www.google.com: `dig www.google.com A`. You may read the man page for `dig` <a href="http://linux.die.net/man/1/dig">here</a>.

<strong>Answer the following questions in a Word document. You will submit a Word document for your final submission:</strong> 

1. The `dig` command sent a DNS query to some DNS server and received a response from that server. What is the IP address that server?
2. The DNS response includes a status. What was the status of the response?
3. The response identified the IP address of www.google.com. What is the reported IP address of www.google.com?
4. How long will the www.google.com IPv4 address be cached?
5. The response listed the nameserver that is authoritative for www.google.com. What is the authoritative name server for google.com?
6. Finally, the response listed the IPv4 address for the google.com nameserver. For each such server what is its IPv4 address?

<span style="color: green"><strong><img src="resources/idea.png" style="width: 12px"> Tips:</strong></span> 
- Be sure to issue the `dig` command from the `client` machine. The IPv4 addresses associated with this exercise begin with `10`. If you are seeing other IPv4 address, then you may be querying the real DNS tree rather than the one in the exercise. While this is harmless and interesting in and of itself, the later steps assume you are using the DNS tree created in this exercise.

## <strong>Topic 2: Understanding Network Topology</strong>

Before doing any ARP spoofing, we want to understand the network topology. Use the topology visualization in this document and use `arp` and `ifconfig` commands to detect MAC and IP addresses for each machine. 

It's highly recommended to view the topology graph that's provided in the introduction. This will help you understand the layout of the nodes.

## Step 2: The Big Picture

<strong>Your task:</strong> Read the questions carefully, and pay attention to which node(s) that each question asks you to enter.

<strong>Answer the following questions.</strong>
1. State the source MAC and IP addresses as well as destination MAC and IP addresses for a packet going from the `client` to the cache.
2. Does the packet travel through the attacker box?
3. State the source MAC and IP addresses as well as destination MAC and IP addresses for a packet going from the `cache` to the authoritative server.
4. Does the packet travel through the attacker box?

## <strong>Topic 3: Using Ettercap</strong>

Ettercap is a powerful tool for network analysis and security cracking. Ettercap has two main strengths:

First, Ettercap is capable of performing man-in-the-middle attacks on many protocols, including ARP and DNS. Ettercap is capable of "filtering" packets, where it intercepts, changes, and resends packets on the fly. This could be as simple as changing all occurrences of the word "buy" to "sell" in all web traffic, or as complex as exchanging the encryption keys in a secure transmission.

Second, Ettercap is capable of fully automating ARP spoofing (also known as ARP poisoning) which is a technique whereby a computer on a switched network is capable of forcing other hosts to send their data to the attacker rather than the actual network gateway or destination. This allows an attacker to force arbitrary users to send their data to him first—after which he can choose to send the traffic on, change it, or throw it away—all without any privileged position on the network other than the complete control of one workstation.

By the inclusion of ARP spoofing, expressive filters, and man-in-the-middle attacks, Ettercap is a one-stop-shop for many network attacks. 

## Step 3: Performing an ARP Spoof

<strong>Your task:</strong> Login to the `attacker` machine. Using ettercap, your objective is to get the DNS query for www.google.com to pass through the `attacker`. Once you've accomplished this and confirmed that the desired traffic is now passing through the `attacker`, record the following: 

1. The command you used.
2. What each option in the command means.
3. State the source MAC and IP addresses as well as destination MAC and IP addresses for a packet going from the cache to the authoritative server.
4. Does the packet travel through the attacker box?
5. If Question 3 from <strong>Step 2</strong> and Question 4 from <strong>Step 3</strong> differ, explain why.

<span style="color: red;"><strong><img src="resources/alert.png" style="width: 12px"> Warning:</strong></span> Do not ARP Poison any control interface `(172.30.x.x)`! Check which interface on attacker node has `10.x.x.x` address and use that interface with `-i` option with `ettercap`, to make sure your poisoning stays on experimental network.

## Step 4: DNS Spoofing

Now that you can ARP spoof on the network, your goal is to actually filter DNS messages returning to the end-user and replace the IP for "www.google.com." with an IP of your choosing (say the attacker: `10.1.2.4`). It's possible to do this with a custom made ettercap filter, but due to some of the intricacies of the DNS protocol, it can get quite complicated. Since the DNS protocol is not the focus of this exercise, we recommend you use a plug-in built into `ettercap` called `dns_spoof`. You can find the syntax for using a plug-in via the ettercap man page, and the config file for `dns_spoof` (located at `/etc/ettercap/etter.dns`) should be self-explanatory. A good example of using ettercap for DNS spoofing is <a href="http://www.thegeekstuff.com/2012/05/ettercap-tutorial/">at this page</a>. (These authors use the graphical interface, although the text mode interface is similar.)

<strong>Your task:</strong> Answer the following questions.
1. The complete command (or steps in the GUI) used to have ettercap forge a DNS message and any necessary configuration files.
2. What malicious things could an attacker do by changing the IP address in a DNS response going to the client? 

## <strong>Topic 4: Implementing DNSSEC</strong>

The final task to prevent this type of attack by cryptographically signing the DNS data. Login to `auth` and follow instructions <a href="https://www.digitalocean.com/community/tutorials/how-to-setup-dnssec-on-an-authoritative-bind-dns-server-2">here</a> to set up DNSSEC. <strong>IMPORTANT: Your zone file is already present at /etc/bind/zones/db.google.com. This is the file you should sign.</strong> Then restart `bind` and try on the auth machine: `dig +dnssec www.google.com A`. You should get a signed response.

## Step 5: Creating Your Implementation

On the cache machine, add the public zone signing key (ZSK) to the list of trust-anchors for the cache. An example how to do this is <a href="https://benctechnicalblog.blogspot.com/2011/01/dnssec-configuring-my-resolver.html">here</a> You can use `dig google.com dnskey` on `auth` to learn what keys to add to the list. You must also add lines to `/etc/bind/named.conf.options` that tell it to use `dnssec` (same ones as in the previous step). Then restart bind on cache and run `dig +dnssec www.google.com A`. You should get a signed response.

Finally, return to the `client` machine and use the command `dig +dnssec www.google.com A` to lookup the IPv4 address for `www.google.com`. You should get a signed response, with AD bit set. 

<strong>Your task:</strong> Submit the following into your Word document.
1. The signed response obtained on the `client` machine.
2. Detailed description of all the steps you took to implement DNSSEC. Make sure to list all commands you typed and all configuration changes you made. 

## <strong>Grading</strong>

This notebook does not have an auto-grader. The Word document that you wrote for this lab will be manually graded by your instructor or TA.

### Stopping the Lab

Once you are done with the lab, click on the "Stop Lab" button below. <strong>This will delete your materialization, which will delete all of the lab's resources.</strong> Your progress is saved automatically in ```saves/``` within the sidebar of your XDC. You may load this lab in the future by clicking "Load Lab" at the top.

In [4]:
# Click the button below to stop the experiment.
def stoplab(button):
    stop_lab(labname, confirm, stop_output)

# Creating the button.
stopButton = widgets.Button(description="Stop Lab")

# Create a confirmation check.
confirm = widgets.Checkbox(
    value=False,
    description='Confirm',
    disabled=False,
    indent=False
)

# Creating an output area.
stop_output = widgets.Output()

# Run the command on click.
stopButton.on_click(stoplab)

# Display the output.
display(confirm, stopButton, stop_output)

Checkbox(value=False, description='Confirm', indent=False)

Button(description='Stop Lab', style=ButtonStyle())

Output()